# 01.02 环境与工程一键启动

## 小节概述

本节把“环境可用”拆成可观察、可复现的检查项，并用一个代码单元完成 VectorAdd 演示工程的配置、编译和运行。你还会查看工程目录与关键文件，理解源码、构建配置和可执行程序之间的关系。

完成本节后，你应能够：

1. 判断 CANN 环境、ASC 编译工具和昇腾 NPU 是否就绪；
2. 说明 <code>vector_add.asc</code> 与 <code>CMakeLists.txt</code> 的作用；
3. 从 Notebook 一键完成干净构建，并从 <code>METRIC</code> 行判断运行是否成功。

开始前请先阅读 [01.01 章节介绍](01.01_chapter_intro.ipynb)。本节按 CANNLab 镜像 <code>cann_9.0.0 py3.11-A2-arm</code>、Notebook 内核 <code>Python 3.11.4 (CANN)</code> 编写。


## 教程内容

### 1. 定位课程与独立构建目录

下面的代码从当前 Notebook 向上查找仓库根目录，因此从课程目录或仓库根目录启动都可以。构建产物统一放到 <code>/tmp</code>，不会写入课程源码目录。


In [ ]:
from pathlib import Path
import getpass
import os
import platform
import shutil
import subprocess
import sys
import tempfile

previous_repo = globals().get('REPO_ROOT')
try:
    start = Path.cwd().resolve()
except FileNotFoundError:
    cached_repo = Path(previous_repo) if previous_repo is not None else None
    if cached_repo is None or not (cached_repo / 'contrib/tutorials/data_structures_compute').is_dir():
        raise RuntimeError(
            '当前内核的工作目录已被清理，请重启内核后从本节第一个代码单元开始运行'
        ) from None
    os.chdir(cached_repo)
    start = cached_repo.resolve()
search_roots = []
if previous_repo is not None:
    search_roots.append(Path(previous_repo).resolve())
search_roots.extend([start, *start.parents])
REPO_ROOT = next(
    (p for p in search_roots if (p / 'contrib/tutorials/data_structures_compute').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('请从 cann-learning-hub 仓库中打开本 Notebook')
os.chdir(REPO_ROOT)

CHAPTER = REPO_ROOT / 'contrib/tutorials/data_structures_compute/01_basic_operations'
DEMO = CHAPTER / 'src/demo'
USER_KEY = f'uid_{os.getuid()}' if hasattr(os, 'getuid') else f'user_{getpass.getuser()}'
USER_TEMP_ROOT = Path(tempfile.gettempdir()) / f'cannlab_data_structures_compute_{USER_KEY}'
WORK_ROOT = USER_TEMP_ROOT / '01_environment_and_project'

print('repository :', REPO_ROOT)
print('chapter    :', CHAPTER)
print('demo       :', DEMO)
print('build root :', WORK_ROOT)


### 2. 检查 CANN、编译工具与 NPU

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>检查项</th><th style='text-align: left;'>为什么需要</th><th style='text-align: left;'>正常现象</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>Python</code></td><td style='text-align: left;'>执行 Notebook 中的工程编排与自检代码</td><td style='text-align: left;'>使用 CANN Python 内核</td></tr>
    <tr><td style='text-align: left;'><code>cmake</code></td><td style='text-align: left;'>读取工程配置并生成构建文件</td><td style='text-align: left;'>可以定位到可执行文件</td></tr>
    <tr><td style='text-align: left;'><code>ASC</code></td><td style='text-align: left;'>编译 <code>.asc</code> 中的 Ascend C Device 代码</td><td style='text-align: left;'>CMake 的 <code>find_package(ASC REQUIRED)</code> 成功</td></tr>
    <tr><td style='text-align: left;'><code>npu-smi</code></td><td style='text-align: left;'>查看当前实例分配到的昇腾设备</td><td style='text-align: left;'><code>npu-smi info</code> 返回设备表格</td></tr>
  </tbody>
</table>

<code>ASC</code> 是 Ascend C 编译语言支持；它通常随 CANN 环境加载，不一定对应一个名为 <code>asc</code> 的独立命令，因此最终以第 4 节 CMake 配置是否成功为准。


In [ ]:
print('Python:', sys.version.split()[0], '-', platform.platform())
for variable in ['ASCEND_HOME_PATH', 'ASCEND_TOOLKIT_HOME', 'LD_LIBRARY_PATH']:
    value = os.environ.get(variable)
    print(f'{variable}:', value if value else 'NOT SET')

for tool in ['cmake', 'npu-smi', 'c++']:
    print(f'{tool}:', shutil.which(tool) or 'NOT FOUND')

npu_smi = shutil.which('npu-smi')
if npu_smi:
    subprocess.run([npu_smi, 'info'], check=False)
else:
    print('npu-smi 不可用：请确认实例已分配 NPU，并选择 CANN Python 内核。')


### 3. 浏览工程结构与关键文件

演示工程只包含两个关键文件。<code>vector_add.asc</code> 同时放置 Host 启动代码与 Device Kernel，便于初学者沿一次运行完整追踪；<code>CMakeLists.txt</code> 声明 ASC/C++ 工程、目标架构和可执行目标。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>文件</th><th style='text-align: left;'>作用</th><th style='text-align: left;'>本节关注点</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>src/demo/vector_add.asc</code></td><td style='text-align: left;'>参数检查、内存管理、Kernel 与 CPU Golden 校验</td><td style='text-align: left;'>确认它是构建输入；实现细节在 01.03 分阶段阅读</td></tr>
    <tr><td style='text-align: left;'><code>src/demo/CMakeLists.txt</code></td><td style='text-align: left;'>配置 ASC/C++ 编译并生成 <code>vector_add</code></td><td style='text-align: left;'>确认目标架构默认值和目标名称</td></tr>
  </tbody>
</table>

下面用 <code>ls</code> 展示源码树，并用 <code>cat</code> 展示完整构建文件；学习者无需手动离开 Notebook 查找文件。


In [ ]:
subprocess.run(['ls', '-la', str(DEMO)], check=True)
print('\n--- CMakeLists.txt ---')
subprocess.run(['cat', str(DEMO / 'CMakeLists.txt')], check=True)
print('\n--- vector_add.asc 前 80 行 ---')
subprocess.run(['sed', '-n', '1,80p', str(DEMO / 'vector_add.asc')], check=True)


### 4. 一键配置、编译并运行

下一单元执行一次完整闭环：删除本节当前用户专属的临时构建目录、运行 CMake 配置、编译 <code>vector_add</code>，再用默认 Shape 启动程序。该目录由系统临时目录与当前用户标识共同确定（支持 UID 的系统使用 UID，否则使用登录用户名），实际路径见上方 <code>build root</code> 输出；<strong>删除范围仅限 <code>WORK_ROOT</code></strong>，课程源码不会被修改。

CMake 参数 <code>-S</code> 指定源码目录，<code>-B</code> 指定构建目录；<code>cmake --build</code> 根据生成的构建文件编译目标。默认 <code>NPU_ARCH=dav-2201</code> 对应本课程的 910B3/A2 验证基线，如需切换目标，必须记录实际硬件并显式传入 <code>-DNPU_ARCH=&lt;target&gt;</code>。


In [ ]:
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
BUILD = WORK_ROOT / 'build'
BUILD.mkdir(parents=True)

steps = [
    ['cmake', '-S', str(DEMO), '-B', str(BUILD)],
    ['cmake', '--build', str(BUILD), '--parallel'],
]
for command in steps:
    print('$', ' '.join(command))
    subprocess.run(command, check=True)

executable = BUILD / 'vector_add'
run_command = [
    str(executable), '--length', '16384', '--block-dim', '8',
    '--tile-count', '8', '--seed', '7', '--warmup', '1', '--iterations', '5',
]
print('$', ' '.join(run_command))
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
print(run_result.stdout)


成功时，末尾 <code>METRIC</code> 行至少应包含 <code>length=16384</code>、<code>block_dim=8</code>、<code>tile_count=8</code>、<code>correctness=PASS</code> 和 <code>max_abs_error=0.000000</code>。<code>avg_kernel_us</code> 是 Host 侧 <code>launch_plus_sync</code> 口径，会随设备负载波动，本节不以固定耗时作为验收条件。

<table style='text-align: left; margin-left: 0;'>
  <thead><tr><th style='text-align: left;'>现象</th><th style='text-align: left;'>优先检查</th><th style='text-align: left;'>处理建议</th></tr></thead>
  <tbody>
    <tr><td style='text-align: left;'><code>find_package(ASC)</code> 失败</td><td style='text-align: left;'>镜像和 Notebook 内核</td><td style='text-align: left;'>重新选择 CANN Python 内核后从第 2 节检查</td></tr>
    <tr><td style='text-align: left;'>找不到 NPU 或 ACL 初始化失败</td><td style='text-align: left;'><code>npu-smi info</code></td><td style='text-align: left;'>确认实例分配了 NPU，而不是仅有 CPU</td></tr>
    <tr><td style='text-align: left;'><code>correctness=FAIL</code></td><td style='text-align: left;'>最早错误和 <code>max_abs_error</code></td><td style='text-align: left;'>不要先比较性能，进入 01.03 检查数据通路</td></tr>
  </tbody>
</table>


## 课后实践

### 5. 编写可复用的工具检查脚本

<strong>独立输入：</strong> 工具名称列表 <code>['cmake', 'npu-smi', 'c++']</code>。

<strong>任务：</strong> 补全下面文件中的 <code>tool_status</code>。当工具存在时返回 <code>FOUND: &lt;绝对路径&gt;</code>，否则返回 <code>MISSING</code>；不得把某个机器上的固定路径写入代码。运行脚本后应输出三行状态。

练习写入当前用户专属的临时课程根目录，与其他用户及教程演示工程解耦。下面先准备目录；下一单元使用 <code>%%writefile</code> 提供待填写文件。


In [ ]:
PRACTICE = USER_TEMP_ROOT / '01_environment_practice'
PRACTICE.mkdir(parents=True, exist_ok=True)
PRACTICE_FILE = PRACTICE / 'environment_check.py'
print('practice file:', PRACTICE_FILE)


In [ ]:
%%writefile {PRACTICE_FILE}
from pathlib import Path
import shutil

TOOLS = ['cmake', 'npu-smi', 'c++']

def tool_status(name):
    path = shutil.which(name)
    # TODO：存在时返回 FOUND: <绝对路径>，否则返回 MISSING。
    return 'TODO'

if __name__ == '__main__':
    for tool in TOOLS:
        print(f'{tool}: {tool_status(tool)}')


In [ ]:
practice_result = subprocess.run(
    [sys.executable, str(PRACTICE_FILE)],
    text=True,
    capture_output=True,
    check=True,
)
print(practice_result.stdout)
source_text = PRACTICE_FILE.read_text(encoding='utf-8')
print('PRACTICE PASS' if "return 'TODO'" not in source_text else 'PRACTICE TODO')


#### 独立完成后查看参考实现

先补全文件并重新运行自检。确认独立完成后，将开关改为 <code>True</code>；答案通过代码单元调用 <code>cat</code> 展示。


In [ ]:
SHOW_ANSWER = False
answer_file = CHAPTER / 'answer/01.02_environment_and_project/environment_check.py'
if SHOW_ANSWER:
    subprocess.run(['cat', str(answer_file)], check=True)
else:
    print('参考实现保持隐藏；独立完成后将 SHOW_ANSWER 改为 True。')


## 本节小结

你已经把环境、源码、构建和运行串成了一条可复现路径：先确认 CANN 内核与 NPU，再从 Notebook 展示关键源码并执行干净构建，最后用精度字段而不是固定耗时判断是否成功。


完成后继续进入 [01.03 VectorAdd 算子实验](01.03_vector_add_operator.ipynb)。
